In [1]:
"""
03 - Train liking-score predictor models with LOO-CV.

Outputs:
- outputs/model_metrics.json: LOO-CV scores per model
- data/trained_models.pkl: fitted Ridge + RF (on full data) for downstream prediction
- outputs/loocv_predictions.csv: actual vs predicted per fold
"""
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.dummy import DummyRegressor

In [3]:
ROOT = Path("__file__" in globals() and __file__ or "notebooks/dummy.py").resolve().parents[1] if "__file__" in globals() else Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"

In [4]:
with open(DATA / "training_data.pkl", "rb") as f:
    td = pickle.load(f)

In [5]:
X, y, names = td["X"], td["y"], td["names"]
print(f"Loaded training data: X={X.shape}, y={y.shape}")

Loaded training data: X=(13, 300), y=(13,)


In [6]:
# L2-normalise embeddings (best practice for cosine-based comparisons)
X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)

In [7]:
# v3 feature: cosine similarity to "high-sentiment centroid" (top-3 flavors)
N_ANCHOR = 5
top_idx = np.argsort(y)[-N_ANCHOR:]
top_centroid = X_norm[top_idx].mean(axis=0)
top_centroid = top_centroid / np.linalg.norm(top_centroid)
sim_to_top = cosine_similarity(X_norm, top_centroid.reshape(1, -1)).flatten()
print(f"Top-3 anchor flavors (highest sentiment): {[names[i] for i in top_idx]}")
print(f"Cosine sim to top centroid: range=[{sim_to_top.min():.3f}, {sim_to_top.max():.3f}]")
print(f"Pearson corr(sim_to_top, y) = {np.corrcoef(sim_to_top, y)[0,1]:.3f}")

Top-3 anchor flavors (highest sentiment): ['milk_tea', 'coconut', 'caramel', 'vanilla', 'mango']
Cosine sim to top centroid: range=[0.205, 0.680]
Pearson corr(sim_to_top, y) = 0.602


In [8]:
# Build feature variants
X_sim_feat = sim_to_top.reshape(-1, 1)  # single feature: similarity to liked anchors

In [9]:
# Define candidate models
def build_pca_ridge(n_components, alpha):
    return Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=42)),
        ("model", Ridge(alpha=alpha)),
    ])

In [10]:
models = {
    "baseline_mean":     ("X",       DummyRegressor(strategy="mean")),
    "ridge_a1":          ("X",       Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))])),
    "ridge_a10":         ("X",       Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=10.0))])),
    "ridge_a100":        ("X",       Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=100.0))])),
    "pca5_ridge_a1":     ("X",       build_pca_ridge(5, 1.0)),
    "pca3_ridge_a1":     ("X",       build_pca_ridge(3, 1.0)),
    "pca2_ridge_a1":     ("X",       build_pca_ridge(2, 1.0)),
    "kernel_ridge_rbf":  ("X",       Pipeline([("scale", StandardScaler()), ("model", KernelRidge(alpha=1.0, kernel="rbf", gamma=0.01))])),
    "random_forest":     ("X",       RandomForestRegressor(n_estimators=200, max_depth=4, random_state=42)),
    "sim_anchor_linear": ("X_sim",   Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=0.1))])),
}

In [11]:
# LOO-CV
loo = LeaveOneOut()
results = {}
preds_records = []

In [12]:
def fresh_model(model_name):
    """Build a fresh unfit estimator (avoid LOO leakage from refitting)."""
    if model_name == "baseline_mean":
        return DummyRegressor(strategy="mean")
    if model_name.startswith("ridge_a"):
        a = float(model_name.split("_a")[1])
        return Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=a))])
    if model_name.startswith("pca"):
        # e.g. pca5_ridge_a1
        parts = model_name.split("_")
        n = int(parts[0].replace("pca", ""))
        a = float(parts[2].replace("a", ""))
        return build_pca_ridge(n, a)
    if model_name == "kernel_ridge_rbf":
        return Pipeline([("scale", StandardScaler()), ("model", KernelRidge(alpha=1.0, kernel="rbf", gamma=0.01))])
    if model_name == "random_forest":
        return RandomForestRegressor(n_estimators=200, max_depth=4, random_state=42)
    if model_name == "sim_anchor_linear":
        return Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=0.1))])
    raise ValueError(model_name)

In [13]:
for model_name, (feat_key, _) in models.items():
    preds = np.zeros(len(y))
    # For sim_anchor, anchor centroid must be recomputed per fold (to avoid leakage)
    for train_idx, test_idx in loo.split(X):
        if feat_key == "X_sim":
            # Recompute anchor from training data only
            train_top = train_idx[np.argsort(y[train_idx])[-N_ANCHOR:]]
            anchor = X_norm[train_top].mean(axis=0)
            anchor = anchor / np.linalg.norm(anchor)
            X_fold = cosine_similarity(X_norm, anchor.reshape(1, -1))
        else:
            X_fold = X

        m_ = fresh_model(model_name)
        m_.fit(X_fold[train_idx], y[train_idx])
        preds[test_idx[0]] = m_.predict(X_fold[test_idx])[0]

    mae = mean_absolute_error(y, preds)
    r2 = r2_score(y, preds)
    rmse = float(np.sqrt(np.mean((y - preds) ** 2)))
    spearman = float(pd.Series(y).corr(pd.Series(preds), method="spearman"))
    results[model_name] = {
        "loocv_mae": float(mae),
        "loocv_r2": float(r2),
        "loocv_rmse": rmse,
        "spearman_corr": spearman,
    }
    for i, (a, p) in enumerate(zip(y, preds)):
        preds_records.append({"model": model_name, "flavor": names[i], "actual": int(a), "predicted": float(p)})

In [14]:
print("\n=== LOO-CV Results ===")
df_metrics = pd.DataFrame(results).T
print(df_metrics.round(3).to_string())


=== LOO-CV Results ===
                   loocv_mae  loocv_r2  loocv_rmse  spearman_corr
baseline_mean         10.372    -0.174      12.976         -1.000
ridge_a1              10.490    -0.311      13.717          0.022
ridge_a10             10.164    -0.240      13.337          0.088
ridge_a100             9.272    -0.061      12.336          0.132
pca5_ridge_a1         10.047    -0.054      12.295          0.036
pca3_ridge_a1         10.519    -0.210      13.174         -0.455
pca2_ridge_a1         10.587    -0.237      13.323         -0.529
kernel_ridge_rbf      32.907    -7.666      35.262          0.237
random_forest         10.231    -0.120      12.678          0.088
sim_anchor_linear     13.044    -0.615      15.222         -0.140


In [15]:
# Save metrics
OUT.mkdir(exist_ok=True)
with open(OUT / "model_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

In [16]:
pred_df = pd.DataFrame(preds_records)
pred_df.to_csv(OUT / "loocv_predictions.csv", index=False)

In [17]:
# Pick best non-baseline model by Spearman (more robust for small-N ranking-style use case)
ranked = sorted(
    (n for n in results if n != "baseline_mean"),
    key=lambda n: results[n]["spearman_corr"],
    reverse=True,
)
best_name = ranked[0]
print(f"\nBest non-baseline model (by Spearman): {best_name}")
print(f"  Spearman={results[best_name]['spearman_corr']:.3f}, R²={results[best_name]['loocv_r2']:.3f}, MAE={results[best_name]['loocv_mae']:.2f}")


Best non-baseline model (by Spearman): kernel_ridge_rbf
  Spearman=0.237, R²=-7.666, MAE=32.91


In [18]:
# Fit final models on FULL data
final_models = {}
feat_key, _ = models[best_name]
final_models["best"] = fresh_model(best_name)
if feat_key == "X_sim":
    final_models["best"].fit(sim_to_top.reshape(-1, 1), y)
else:
    final_models["best"].fit(X, y)

In [19]:
# Also keep sim_anchor_linear as the conservative default (interpretable, low-variance)
final_models["sim_anchor_linear"] = fresh_model("sim_anchor_linear")
final_models["sim_anchor_linear"].fit(sim_to_top.reshape(-1, 1), y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scale', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None


In [20]:
with open(DATA / "trained_models.pkl", "wb") as f:
    pickle.dump({
        "models": final_models,
        "best_name": best_name,
        "best_feature_key": feat_key,
        "top_anchor_centroid": top_centroid,
        "top_anchor_flavors": [names[i] for i in top_idx],
        "training_y_mean": float(y.mean()),
        "training_y_std": float(y.std()),
        "training_y_min": float(y.min()),
        "training_y_max": float(y.max()),
    }, f)
print(f"Saved trained models: {list(final_models.keys())}")
print(f"Outputs: outputs/model_metrics.json, outputs/loocv_predictions.csv, data/trained_models.pkl")

Saved trained models: ['best', 'sim_anchor_linear']
Outputs: outputs/model_metrics.json, outputs/loocv_predictions.csv, data/trained_models.pkl
